In [ ]:
import os
import re
import sys
import subprocess
import warnings
import random
import time
import json
import shutil
import platform
import hashlib
import importlib.metadata as importlib_metadata
from pathlib import Path
from itertools import combinations

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REQUIRED_PACKAGES = [
    "openpyxl>=3.1",
    "optuna>=3.6",
    "xgboost>=2.0",
    "catboost>=1.2",
    "statsmodels>=0.14",
]

if IN_COLAB:
    print("[Bilgi] Colab algılandı. Gerekli paketler kontrol edilip kuruluyor...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *REQUIRED_PACKAGES]
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import friedmanchisquare, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests

from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    KFold, RepeatedKFold, GroupKFold, train_test_split
)
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error
)
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.neural_network import MLPRegressor

import xgboost as xgb
from catboost import CatBoostRegressor
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(SEED)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.dpi": 120,
})

print(f"Python       : {platform.python_version()}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"scikit-learn : {importlib_metadata.version('scikit-learn')}")
print(f"PyTorch      : {torch.__version__}")
print(f"XGBoost      : {xgb.__version__}")
print(f"CatBoost     : {importlib_metadata.version('catboost')}")
print(f"Optuna       : {optuna.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

In [ ]:


CODE_VERSION = "v27.0"

FAST_MODE = False
RESUME_FROM_CHECKPOINT = True
RUN_ABLATION = False 
RUN_FINAL_MODELS = True

TARGET = "HI"

DATA_CANDIDATES = [
    "arpa veriseti.xlsx",
    "/content/drive/MyDrive/barley_project/arpa veriseti.xlsx",
]

MANUAL_GROUP_COLUMN = None
LEAKAGE_POLICY = "legacy"  
FINAL_FEATURE_SET_KEY = "A2_SAFE_ENGINEERED"

OUTER_SPLITS = 5
OUTER_REPEATS = 1 if FAST_MODE else 2
INNER_SPLITS = 3

TRIALS_FAST = {
    "Ridge": 4,
    "SVR": 4,
    "Random Forest": 4,
    "Extra Trees": 4,
    "Gradient Boosting": 4,
    "XGBoost": 4,
    "CatBoost": 4,
    "Gaussian Process Regression": 3,
    "MLP": 4,
    "KAN": 3,
}
TRIALS_FULL = {
    "Ridge": 10,
    "SVR": 15,
    "Random Forest": 12,
    "Extra Trees": 12,
    "Gradient Boosting": 12,
    "XGBoost": 15,
    "CatBoost": 15,
    "Gaussian Process Regression": 8,
    "MLP": 10,
    "KAN": 8,
}
N_TRIALS = TRIALS_FAST if FAST_MODE else TRIALS_FULL
ABLATION_TRIALS = {name: max(3, trials // 2) for name, trials in N_TRIALS.items()}

ABLATION_MODELS = ["Ridge", "SVR", "Random Forest", "XGBoost"]
CORE_MODELS = [
    "Ridge",
    "SVR",
    "Random Forest",
    "Extra Trees",
    "Gradient Boosting",
    "XGBoost",
    "CatBoost",
    "Gaussian Process Regression",
    "MLP",
    "KAN",
]

OUTPUT_DIR_NAME = "tez_analiz_v27_SGW_SPW_dahil_outputs"

In [ ]:


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

def locate_data_file(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return None

DATA_PATH = locate_data_file(DATA_CANDIDATES)

if DATA_PATH is None and IN_COLAB:
    print("[Uyarı] Veri dosyası Drive'da bulunamadı. Lütfen Excel dosyasını seçin.")
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError("Excel dosyası yüklenmedi.")
    DATA_PATH = next(iter(uploaded.keys()))

if DATA_PATH is None:
    raise FileNotFoundError(
        "Veri dosyası bulunamadı. DATA_CANDIDATES listesine doğru yolu ekleyin."
    )

if IN_COLAB and os.path.exists("/content/drive/MyDrive/barley_project"):
    OUTPUT_DIR = Path("/content/drive/MyDrive/barley_project") / OUTPUT_DIR_NAME
else:
    OUTPUT_DIR = Path(OUTPUT_DIR_NAME)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"[Bilgi] Veri dosyası  : {DATA_PATH}")
print(f"[Bilgi] Çıktı klasörü: {OUTPUT_DIR}")

df_raw = pd.read_excel(DATA_PATH, na_values=[" -", "-", " ", ""])
df_raw.columns = [str(c).strip() for c in df_raw.columns]

if TARGET not in df_raw.columns:
    raise KeyError(f"Hedef sütun bulunamadı: {TARGET}")

df = df_raw.dropna(subset=[TARGET]).reset_index(drop=True).copy()

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

print(f"Gözlem sayısı : {len(df)}")
print(f"Sütun sayısı  : {df.shape[1]}")
display(df.head())

In [ ]:


REF_FEATURES_ORIGINAL = [
    "DFL", "PH (cm)", "SL (mm)", "RNS", "NGS",
    "SGW (g)", "SPW (g)", "TGW (g)", "NT"
]

ALL_14_FEATURES_ORIGINAL = [
    "DFL", "PH (cm)", "FLA (cm2)", "SL (mm)", "RNS",
    "SPW (g)", "NT", "TR", "KR", "KVZ", "KLÇ",
    "NGS", "SGW (g)", "TGW (g)"
]

CATEGORICAL_CANDIDATES = ["TR", "KR", "KVZ", "KLÇ"]

missing_required = sorted(
    set(ALL_14_FEATURES_ORIGINAL + [TARGET]) - set(df.columns)
)
if missing_required:
    raise KeyError(
        "Beklenen sütunlar veri setinde bulunamadı: " + ", ".join(missing_required)
    )

schema_rows = []
for col in ALL_14_FEATURES_ORIGINAL:
    unique_values = df[col].dropna().unique()
    unique_preview = sorted(map(str, unique_values))[:15]
    schema_rows.append({
        "Sütun": col,
        "dtype": str(df[col].dtype),
        "Eksik": int(df[col].isna().sum()),
        "Tekil_değer": int(df[col].nunique(dropna=True)),
        "Örnek_değerler": ", ".join(unique_preview),
        "Önerilen_tür": (
            "Nominal kategorik"
            if col in CATEGORICAL_CANDIDATES
            else "Sayısal"
        ),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df)
schema_df.to_csv(OUTPUT_DIR / "veri_sozlugu_kategorik_denetim.csv", index=False)

CATEGORICAL_FEATURES = [
    c for c in CATEGORICAL_CANDIDATES if c in df.columns
]

ORDINAL_SCHEMA = {}

if ORDINAL_SCHEMA:
    raise NotImplementedError(
        "Bu sürüm nominal one-hot encoding kullanır. Ordinal şema eklenirse "
        "preprocessor hücresi buna göre güncellenmelidir."
    )

In [ ]:


def safe_divide(a, b):
    a = pd.to_numeric(a, errors="coerce").astype(float)
    b = pd.to_numeric(b, errors="coerce").astype(float)
    return a / b.replace(0, np.nan)

FORMULA_CANDIDATES = {
    "SGW/SPW": (
        ["SGW (g)", "SPW (g)"],
        lambda d: safe_divide(d["SGW (g)"], d["SPW (g)"])
    ),
    "100*SGW/SPW": (
        ["SGW (g)", "SPW (g)"],
        lambda d: 100.0 * safe_divide(d["SGW (g)"], d["SPW (g)"])
    ),
    "100*SGW/(SGW+SPW)": (
        ["SGW (g)", "SPW (g)"],
        lambda d: 100.0 * safe_divide(
            d["SGW (g)"], d["SGW (g)"] + d["SPW (g)"]
        )
    ),
    "100*(NGS*TGW/1000)/SPW": (
        ["NGS", "TGW (g)", "SPW (g)"],
        lambda d: 100.0 * safe_divide(
            pd.to_numeric(d["NGS"], errors="coerce")
            * pd.to_numeric(d["TGW (g)"], errors="coerce") / 1000.0,
            d["SPW (g)"]
        )
    ),
}

def audit_hi_formulas(data: pd.DataFrame, target: str) -> pd.DataFrame:
    rows = []
    y_all = pd.to_numeric(data[target], errors="coerce").astype(float)

    for formula_name, (components, formula_fn) in FORMULA_CANDIDATES.items():
        if not set(components).issubset(data.columns):
            continue

        candidate = formula_fn(data)
        valid = y_all.notna() & candidate.notna() & np.isfinite(candidate)
        y = y_all.loc[valid].to_numpy()
        x = candidate.loc[valid].to_numpy().reshape(-1, 1)

        if len(y) < 10 or np.std(x) == 0:
            continue

        corr = float(np.corrcoef(x.ravel(), y)[0, 1])
        calibrator = LinearRegression().fit(x, y)
        y_hat = calibrator.predict(x)
        rmse = float(np.sqrt(mean_squared_error(y, y_hat)))
        nrmse = rmse / (float(np.std(y, ddof=1)) + 1e-12)
        calibrated_r2 = float(r2_score(y, y_hat))

        strong_proxy = (
            abs(corr) >= 0.98
            and calibrated_r2 >= 0.95
            and nrmse <= 0.25
        )

        rows.append({
            "Aday_formül": formula_name,
            "Bileşenler": ", ".join(components),
            "Pearson_r": corr,
            "Kalibre_R2": calibrated_r2,
            "Kalibre_NRMSE": nrmse,
            "Eğim": float(calibrator.coef_[0]),
            "Sabit": float(calibrator.intercept_),
            "Güçlü_matematiksel_vekil_riski": bool(strong_proxy),
        })

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values(
        "Kalibre_R2", ascending=False
    ).reset_index(drop=True)

leakage_audit_df = audit_hi_formulas(df, TARGET)
display(leakage_audit_df)
leakage_audit_df.to_csv(
    OUTPUT_DIR / "HI_matematiksel_iliskiler_denetimi.csv", index=False
)

DIRECT_COMPONENTS_STRICT = ["SGW (g)", "SPW (g)"]
HIGH_RISK_ENGINEERED = [
    "SGW_to_SPW",
    "NGS_x_TGW",
    "SGW_to_NGS",
    "SGW_to_FLA",
]

auto_formula_components = set()
if not leakage_audit_df.empty:
    risky_rows = leakage_audit_df[
        leakage_audit_df["Güçlü_matematiksel_vekil_riski"]
    ]
    for component_text in risky_rows["Bileşenler"]:
        auto_formula_components.update(
            [c.strip() for c in component_text.split(",")]
        )

if LEAKAGE_POLICY == "strict":
    RAW_EXCLUSIONS = sorted(
        set(DIRECT_COMPONENTS_STRICT) | auto_formula_components
    )
elif LEAKAGE_POLICY == "audit":
    RAW_EXCLUSIONS = sorted(auto_formula_components)
elif LEAKAGE_POLICY == "legacy":
    RAW_EXCLUSIONS = []
else:
    raise ValueError("LEAKAGE_POLICY: strict, audit veya legacy olmalıdır.")

print(f"[Bilgi] Sızıntı politikası: {LEAKAGE_POLICY}")
print(f"[Bilgi] Ana analizden çıkarılan ham özellikler: {RAW_EXCLUSIONS}")

In [ ]:


def infer_group_column(data: pd.DataFrame, manual=None):
    if manual is not None:
        if manual not in data.columns:
            raise KeyError(f"MANUAL_GROUP_COLUMN bulunamadı: {manual}")
        return manual

    preferred = [
        "OTGB NO", "OTGB_NO", "Genotype", "Genotip",
        "Line", "Hat", "Accession"
    ]
    for col in preferred:
        if col in data.columns:
            n_unique = data[col].nunique(dropna=True)
            if OUTER_SPLITS <= n_unique < len(data):
                return col

    return None

GROUP_COLUMN = infer_group_column(df, MANUAL_GROUP_COLUMN)
groups = None if GROUP_COLUMN is None else df[GROUP_COLUMN].astype(str).to_numpy()

if GROUP_COLUMN is None:
    print(
        "[Bilgi] Grup sütunu kullanılmadı. RepeatedKFold uygulanacak.\n"
        "Aynı genotipe ait tekrarlı satırlar varsa MANUAL_GROUP_COLUMN ayarlayın."
    )
else:
    print(
        f"[Bilgi] Grup-korumalı CV kullanılacak: {GROUP_COLUMN} "
        f"({df[GROUP_COLUMN].nunique()} grup)"
    )

In [ ]:


SAFE_ENGINEERED_CANDIDATES = [
    "NT_to_PH",
    "PH_to_DFL",
    "RNS_to_SL",
    "FLA_to_SL",
    "NGS_to_RNS",
    "FLA_to_NGS",
    "PH_x_NT",
    "RNS_x_SL",
]

def available_safe_engineered_names(base_features):
    base = set(base_features)
    required_map = {
        "NT_to_PH": {"NT", "PH (cm)"},
        "PH_to_DFL": {"PH (cm)", "DFL"},
        "RNS_to_SL": {"RNS", "SL (mm)"},
        "FLA_to_SL": {"FLA (cm2)", "SL (mm)"},
        "NGS_to_RNS": {"NGS", "RNS"},
        "FLA_to_NGS": {"FLA (cm2)", "NGS"},
        "PH_x_NT": {"PH (cm)", "NT"},
        "RNS_x_SL": {"RNS", "SL (mm)"},
    }
    return [
        name for name, required in required_map.items()
        if required.issubset(base)
    ]

class AgronomicFeatureBuilder(BaseEstimator, TransformerMixin):
    def __init__(self, base_features, add_engineered=False):
        self.base_features = tuple(base_features)
        self.add_engineered = add_engineered

    def fit(self, X, y=None):
        missing = sorted(set(self.base_features) - set(X.columns))
        if missing:
            raise KeyError(f"Eksik model girdileri: {missing}")
        self.output_features_ = list(self.base_features)
        if self.add_engineered:
            self.output_features_.extend(
                available_safe_engineered_names(self.base_features)
            )
        return self

    @staticmethod
    def _num(series):
        return pd.to_numeric(series, errors="coerce").astype(float)

    @staticmethod
    def _div(a, b):
        return a / b.replace(0, np.nan)

    def transform(self, X):
        out = X.loc[:, list(self.base_features)].copy()

        for col in out.columns:
            if col not in CATEGORICAL_FEATURES:
                out[col] = self._num(out[col])

        if not self.add_engineered:
            return out

        if {"NT", "PH (cm)"}.issubset(out.columns):
            out["NT_to_PH"] = self._div(
                self._num(out["NT"]), self._num(out["PH (cm)"])
            )
        if {"PH (cm)", "DFL"}.issubset(out.columns):
            out["PH_to_DFL"] = self._div(
                self._num(out["PH (cm)"]), self._num(out["DFL"])
            )
        if {"RNS", "SL (mm)"}.issubset(out.columns):
            out["RNS_to_SL"] = self._div(
                self._num(out["RNS"]), self._num(out["SL (mm)"])
            )
            out["RNS_x_SL"] = (
                self._num(out["RNS"]) * self._num(out["SL (mm)"])
            )
        if {"FLA (cm2)", "SL (mm)"}.issubset(out.columns):
            out["FLA_to_SL"] = self._div(
                self._num(out["FLA (cm2)"]), self._num(out["SL (mm)"])
            )
        if {"NGS", "RNS"}.issubset(out.columns):
            out["NGS_to_RNS"] = self._div(
                self._num(out["NGS"]), self._num(out["RNS"])
            )
        if {"FLA (cm2)", "NGS"}.issubset(out.columns):
            out["FLA_to_NGS"] = self._div(
                self._num(out["FLA (cm2)"]), self._num(out["NGS"])
            )
        if {"PH (cm)", "NT"}.issubset(out.columns):
            out["PH_x_NT"] = (
                self._num(out["PH (cm)"]) * self._num(out["NT"])
            )

        return out.replace([np.inf, -np.inf], np.nan)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.output_features_, dtype=object)

In [ ]:


class PCAKMeansAugmenter(BaseEstimator, TransformerMixin):
    
    def __init__(
        self,
        n_components=4,
        n_clusters=7,
        random_state=SEED,
        include_original=True,
    ):
        self.n_components = n_components
        self.n_clusters = n_clusters
        self.random_state = random_state
        self.include_original = include_original

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_components = max(
            1, min(self.n_components, X.shape[1], X.shape[0] - 1)
        )
        n_clusters = max(
            2, min(self.n_clusters, X.shape[0] - 1)
        )

        self.pca_ = PCA(
            n_components=n_components,
            random_state=self.random_state,
        )
        self.kmeans_ = KMeans(
            n_clusters=n_clusters,
            n_init=20,
            random_state=self.random_state,
        )
        self.pca_.fit(X)
        self.kmeans_.fit(X)
        self.n_clusters_ = n_clusters
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        pcs = self.pca_.transform(X)
        labels = self.kmeans_.predict(X)
        cluster_ohe = np.eye(self.n_clusters_, dtype=float)[labels]
        pieces = [pcs, cluster_ohe]
        if self.include_original:
            pieces.insert(0, X)
        return np.hstack(pieces)

In [ ]:


def remove_exclusions(features, exclusions):
    return [f for f in features if f not in set(exclusions)]

REF_FEATURES_SAFE = remove_exclusions(
    REF_FEATURES_ORIGINAL, RAW_EXCLUSIONS
)
ALL_14_FEATURES_SAFE = remove_exclusions(
    ALL_14_FEATURES_ORIGINAL, RAW_EXCLUSIONS
)

FEATURE_SPECS = {
    "A0_REF9_SAFE": {
        "base_features": REF_FEATURES_SAFE,
        "add_engineered": False,
        "add_pca_kmeans": False,
        "description": "Referans özellikler; riskli ham bileşenler çıkarıldı",
    },
    "A1_ALL14_SAFE": {
        "base_features": ALL_14_FEATURES_SAFE,
        "add_engineered": False,
        "add_pca_kmeans": False,
        "description": "14 temel özellik; riskli ham bileşenler çıkarıldı",
    },
    "A2_SAFE_ENGINEERED": {
        "base_features": ALL_14_FEATURES_SAFE,
        "add_engineered": True,
        "add_pca_kmeans": False,
        "description": "A1 + güvenli agronomik özellik mühendisliği",
    },
    "A3_SAFE_ENGINEERED_PCAKMEANS": {
        "base_features": ALL_14_FEATURES_SAFE,
        "add_engineered": True,
        "add_pca_kmeans": True,
        "description": "A2 + fold-içi PCA ve K-Means özellikleri",
    },
}

feature_spec_rows = []
for key, spec in FEATURE_SPECS.items():
    derived = (
        available_safe_engineered_names(spec["base_features"])
        if spec["add_engineered"] else []
    )
    feature_spec_rows.append({
        "Kod": key,
        "Ham_özellik_sayısı": len(spec["base_features"]),
        "Türetilmiş_özellik_sayısı": len(derived),
        "PCA_KMeans": spec["add_pca_kmeans"],
        "Açıklama": spec["description"],
        "Ham_özellikler": ", ".join(spec["base_features"]),
        "Türetilmiş_özellikler": ", ".join(derived),
    })

feature_specs_df = pd.DataFrame(feature_spec_rows)
display(feature_specs_df)
feature_specs_df.to_csv(
    OUTPUT_DIR / "kontrollu_ablation_ozellik_setleri.csv", index=False
)

if FINAL_FEATURE_SET_KEY not in FEATURE_SPECS:
    raise KeyError(f"FINAL_FEATURE_SET_KEY geçersiz: {FINAL_FEATURE_SET_KEY}")

In [ ]:


class KANLinear(nn.Module):
    def __init__(
        self,
        in_features,
        out_features,
        grid_size=5,
        spline_order=3,
        grid_range=(-3.0, 3.0),
    ):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.grid_size = grid_size
        self.spline_order = spline_order

        h = (grid_range[1] - grid_range[0]) / grid_size
        grid = (
            torch.arange(
                -spline_order,
                grid_size + spline_order + 1,
                dtype=torch.float32,
            ) * h + grid_range[0]
        )
        self.register_buffer(
            "grid", grid.expand(in_features, -1).contiguous()
        )

        self.base_weight = nn.Parameter(
            torch.empty(out_features, in_features)
        )
        self.spline_weight = nn.Parameter(
            torch.empty(
                out_features,
                in_features,
                grid_size + spline_order,
            )
        )
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.kaiming_uniform_(
            self.base_weight, a=np.sqrt(5)
        )
        nn.init.normal_(
            self.spline_weight, mean=0.0, std=0.02
        )

    def b_splines(self, x):
        if x.ndim != 2 or x.size(1) != self.in_features:
            raise ValueError("KANLinear girdi boyutu uyumsuz.")

        grid = self.grid
        x_expanded = x.unsqueeze(-1)

        bases = (
            (x_expanded >= grid[:, :-1])
            & (x_expanded < grid[:, 1:])
        ).to(x.dtype)

        for k in range(1, self.spline_order + 1):
            left_num = x_expanded - grid[:, :-(k + 1)]
            left_den = (
                grid[:, k:-1] - grid[:, :-(k + 1)]
            ).clamp_min(1e-12)

            right_num = grid[:, k + 1:] - x_expanded
            right_den = (
                grid[:, k + 1:] - grid[:, 1:-k]
            ).clamp_min(1e-12)

            bases = (
                left_num / left_den * bases[:, :, :-1]
                + right_num / right_den * bases[:, :, 1:]
            )

        return bases.contiguous()

    def forward(self, x):
        base_output = F.linear(F.silu(x), self.base_weight)
        spline_basis = self.b_splines(x).reshape(x.size(0), -1)
        spline_weight = self.spline_weight.reshape(
            self.out_features, -1
        )
        spline_output = F.linear(spline_basis, spline_weight)
        return base_output + spline_output


class SplineKANNet(nn.Module):
    def __init__(
        self,
        input_dim,
        grid_size=5,
        spline_order=3,
    ):
        super().__init__()
        
        
        self.output_layer = KANLinear(
            input_dim, 1,
            grid_size=grid_size,
            spline_order=spline_order,
        )

    def forward(self, x):
        return self.output_layer(x).squeeze(-1)


class TorchKANRegressor(RegressorMixin, BaseEstimator):
    def __init__(
        self,
        grid_size=5,
        spline_order=3,
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=32,
        max_epochs=180,
        patience=25,
        validation_fraction=0.15,
        random_state=SEED,
        device="auto",
        verbose=False,
    ):
        self.grid_size = grid_size
        self.spline_order = spline_order
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.validation_fraction = validation_fraction
        self.random_state = random_state
        self.device = device
        self.verbose = verbose

    def _resolve_device(self):
        if self.device == "auto":
            return torch.device(
                "cuda" if torch.cuda.is_available() else "cpu"
            )
        return torch.device(self.device)

    def fit(self, X, y):
        seed_everything(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1)

        if len(X) < 20:
            raise ValueError("KAN eğitimi için gözlem sayısı çok düşük.")

        self.n_features_in_ = X.shape[1]
        self.device_ = self._resolve_device()

        X_train, X_val, y_train, y_val = train_test_split(
            X, y,
            test_size=self.validation_fraction,
            random_state=self.random_state,
        )

        train_ds = TensorDataset(
            torch.from_numpy(X_train),
            torch.from_numpy(y_train),
        )
        train_loader = DataLoader(
            train_ds,
            batch_size=min(self.batch_size, len(train_ds)),
            shuffle=True,
            generator=torch.Generator().manual_seed(
                self.random_state
            ),
        )

        X_val_t = torch.from_numpy(X_val).to(self.device_)
        y_val_t = torch.from_numpy(y_val).to(self.device_)

        self.model_ = SplineKANNet(
            input_dim=self.n_features_in_,
            grid_size=self.grid_size,
            spline_order=self.spline_order,
        ).to(self.device_)

        optimizer = torch.optim.AdamW(
            self.model_.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )
        loss_fn = nn.MSELoss()

        best_loss = np.inf
        best_state = None
        wait = 0
        self.history_ = {"train_loss": [], "val_loss": []}

        for epoch in range(self.max_epochs):
            self.model_.train()
            batch_losses = []

            for xb, yb in train_loader:
                xb = xb.to(self.device_)
                yb = yb.to(self.device_)

                optimizer.zero_grad(set_to_none=True)
                pred = self.model_(xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model_.parameters(), max_norm=5.0
                )
                optimizer.step()
                batch_losses.append(float(loss.detach().cpu()))

            self.model_.eval()
            with torch.no_grad():
                val_pred = self.model_(X_val_t)
                val_loss = float(
                    loss_fn(val_pred, y_val_t).detach().cpu()
                )

            train_loss = float(np.mean(batch_losses))
            self.history_["train_loss"].append(train_loss)
            self.history_["val_loss"].append(val_loss)

            if val_loss < best_loss - 1e-6:
                best_loss = val_loss
                best_state = {
                    k: v.detach().cpu().clone()
                    for k, v in self.model_.state_dict().items()
                }
                wait = 0
            else:
                wait += 1

            if wait >= self.patience:
                break

        if best_state is not None:
            self.model_.load_state_dict(best_state)

        self.best_validation_loss_ = best_loss
        self.n_epochs_ = len(self.history_["train_loss"])
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        self.model_.eval()
        preds = []

        with torch.no_grad():
            for start in range(0, len(X), 512):
                xb = torch.from_numpy(
                    X[start:start + 512]
                ).to(self.device_)
                preds.append(
                    self.model_(xb).detach().cpu().numpy()
                )

        return np.concatenate(preds).reshape(-1)

In [ ]:


def make_one_hot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )

def get_feature_columns(spec):
    base_features = list(spec["base_features"])
    engineered = (
        available_safe_engineered_names(base_features)
        if spec["add_engineered"] else []
    )
    categorical = [
        c for c in base_features
        if c in CATEGORICAL_FEATURES
    ]
    numeric = [
        c for c in base_features
        if c not in categorical
    ] + engineered
    return numeric, categorical

def make_preprocessor(spec):
    numeric_features, categorical_features = get_feature_columns(spec)

    transformers = []
    if numeric_features:
        numeric_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(
            ("numeric", numeric_pipe, numeric_features)
        )

    if categorical_features:
        categorical_pipe = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ])
        transformers.append(
            ("categorical", categorical_pipe, categorical_features)
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        sparse_threshold=0.0,
        verbose_feature_names_out=False,
    )


class SklearnCatBoostRegressor(RegressorMixin, BaseEstimator):
    """CatBoost'u güncel sklearn tag/clone sözleşmesine uyarlayan ince wrapper."""
    def __init__(
        self,
        iterations=350,
        learning_rate=0.03,
        depth=4,
        l2_leaf_reg=3.0,
        random_strength=1.0,
        loss_function="RMSE",
        random_seed=SEED,
        verbose=0,
        allow_writing_files=False,
        thread_count=2,
    ):
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.l2_leaf_reg = l2_leaf_reg
        self.random_strength = random_strength
        self.loss_function = loss_function
        self.random_seed = random_seed
        self.verbose = verbose
        self.allow_writing_files = allow_writing_files
        self.thread_count = thread_count

    def fit(self, X, y):
        self.model_ = CatBoostRegressor(
            iterations=self.iterations,
            learning_rate=self.learning_rate,
            depth=self.depth,
            l2_leaf_reg=self.l2_leaf_reg,
            random_strength=self.random_strength,
            loss_function=self.loss_function,
            random_seed=self.random_seed,
            verbose=self.verbose,
            allow_writing_files=self.allow_writing_files,
            thread_count=self.thread_count,
        )
        self.model_.fit(X, y)
        self.n_features_in_ = np.asarray(X).shape[1]
        return self

    def predict(self, X):
        return np.asarray(self.model_.predict(X)).reshape(-1)


def make_base_regressor(model_name, seed):
    if model_name == "Ridge":
        return Ridge(alpha=1.0)

    if model_name == "SVR":
        return SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale")

    if model_name == "Random Forest":
        return RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=seed,
            n_jobs=2,
        )

    if model_name == "Extra Trees":
        return ExtraTreesRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=seed,
            n_jobs=2,
        )

    if model_name == "Gradient Boosting":
        return GradientBoostingRegressor(
            n_estimators=250,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=3,
            random_state=seed,
        )

    if model_name == "XGBoost":
        return xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=350,
            learning_rate=0.03,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.0,
            reg_lambda=1.0,
            random_state=seed,
            n_jobs=2,
            verbosity=0,
            tree_method="hist",
        )

    if model_name == "CatBoost":
        return SklearnCatBoostRegressor(
            iterations=350,
            learning_rate=0.03,
            depth=4,
            l2_leaf_reg=3.0,
            random_strength=1.0,
            loss_function="RMSE",
            random_seed=seed,
            verbose=0,
            allow_writing_files=False,
            thread_count=2,
        )

    if model_name == "Gaussian Process Regression":
        kernel = (
            ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=1.0, nu=2.5)
            + WhiteKernel(noise_level=1.0)
        )
        return GaussianProcessRegressor(
            kernel=kernel,
            alpha=1e-6,
            normalize_y=False,
            random_state=seed,
            optimizer=None,
            n_restarts_optimizer=0,
        )

    if model_name == "MLP":
        return MLPRegressor(
            hidden_layer_sizes=(32,),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=800 if not FAST_MODE else 400,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=30,
            random_state=seed,
        )

    if model_name == "KAN":
        return TorchKANRegressor(
            grid_size=5,
            spline_order=3,
            lr=1e-3,
            weight_decay=1e-4,
            batch_size=128,
            max_epochs=140 if not FAST_MODE else 60,
            patience=20 if not FAST_MODE else 10,
            random_state=seed,
            device="auto",
            verbose=False,
        )

    raise KeyError(f"Tanımsız model: {model_name}")

def make_model_pipeline(model_name, feature_spec, seed):
    regressor = make_base_regressor(model_name, seed)

    target_scaled_model = TransformedTargetRegressor(
        regressor=regressor,
        transformer=StandardScaler(),
    )

    pca_kmeans_step = (
        PCAKMeansAugmenter(
            n_components=4,
            n_clusters=7,
            random_state=seed,
            include_original=True,
        )
        if feature_spec["add_pca_kmeans"]
        else "passthrough"
    )

    return Pipeline([
        (
            "features",
            AgronomicFeatureBuilder(
                base_features=feature_spec["base_features"],
                add_engineered=feature_spec["add_engineered"],
            ),
        ),
        ("preprocess", make_preprocessor(feature_spec)),
        ("pca_kmeans", pca_kmeans_step),
        ("model", target_scaled_model),
    ])

In [ ]:


def sample_hyperparams(trial, model_name):
    prefix = "model__regressor__"

    if model_name == "Ridge":
        return {
            prefix + "alpha": trial.suggest_float(
                "alpha", 1e-4, 1e3, log=True
            )
        }

    if model_name == "SVR":
        return {
            prefix + "C": trial.suggest_float(
                "C", 1e-2, 1e3, log=True
            ),
            prefix + "epsilon": trial.suggest_float(
                "epsilon", 1e-3, 0.5, log=True
            ),
            prefix + "gamma": trial.suggest_float(
                "gamma", 1e-4, 1.0, log=True
            ),
        }

    if model_name in {"Random Forest", "Extra Trees"}:
        return {
            prefix + "n_estimators": trial.suggest_int(
                "n_estimators", 200, 500, step=50
            ),
            prefix + "max_depth": trial.suggest_categorical(
                "max_depth", [None, 4, 6, 10, 16]
            ),
            prefix + "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf", 1, 6
            ),
            prefix + "max_features": trial.suggest_float(
                "max_features", 0.4, 1.0
            ),
        }

    if model_name == "Gradient Boosting":
        return {
            prefix + "n_estimators": trial.suggest_int(
                "n_estimators", 100, 400, step=50
            ),
            prefix + "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.15, log=True
            ),
            prefix + "max_depth": trial.suggest_int(
                "max_depth", 1, 4
            ),
            prefix + "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf", 2, 10
            ),
            prefix + "subsample": trial.suggest_float(
                "subsample", 0.6, 1.0
            ),
        }

    if model_name == "XGBoost":
        return {
            prefix + "n_estimators": trial.suggest_int(
                "n_estimators", 100, 450, step=50
            ),
            prefix + "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.15, log=True
            ),
            prefix + "max_depth": trial.suggest_int(
                "max_depth", 2, 6
            ),
            prefix + "min_child_weight": trial.suggest_int(
                "min_child_weight", 1, 8
            ),
            prefix + "subsample": trial.suggest_float(
                "subsample", 0.6, 1.0
            ),
            prefix + "colsample_bytree": trial.suggest_float(
                "colsample_bytree", 0.5, 1.0
            ),
            prefix + "reg_alpha": trial.suggest_float(
                "reg_alpha", 1e-5, 3.0, log=True
            ),
            prefix + "reg_lambda": trial.suggest_float(
                "reg_lambda", 1e-3, 20.0, log=True
            ),
        }

    if model_name == "CatBoost":
        return {
            prefix + "iterations": trial.suggest_int(
                "iterations", 100, 450, step=50
            ),
            prefix + "learning_rate": trial.suggest_float(
                "learning_rate", 0.01, 0.15, log=True
            ),
            prefix + "depth": trial.suggest_int(
                "depth", 3, 7
            ),
            prefix + "l2_leaf_reg": trial.suggest_float(
                "l2_leaf_reg", 1e-2, 20.0, log=True
            ),
            prefix + "random_strength": trial.suggest_float(
                "random_strength", 1e-3, 5.0, log=True
            ),
        }

    if model_name == "Gaussian Process Regression":
        return {
            prefix + "alpha": trial.suggest_float(
                "alpha", 1e-8, 1e-2, log=True
            ),
            prefix + "kernel__k1__k1__constant_value": trial.suggest_float(
                "kernel_constant", 1e-2, 1e2, log=True
            ),
            prefix + "kernel__k1__k2__length_scale": trial.suggest_float(
                "matern_length_scale", 1e-2, 1e2, log=True
            ),
            prefix + "kernel__k2__noise_level": trial.suggest_float(
                "white_noise", 1e-4, 2.0, log=True
            ),
        }

    if model_name == "MLP":
        architecture = trial.suggest_categorical(
            "architecture", ["16", "32", "64", "32-16", "64-32"]
        )
        hidden_map = {
            "16": (16,),
            "32": (32,),
            "64": (64,),
            "32-16": (32, 16),
            "64-32": (64, 32),
        }
        return {
            prefix + "hidden_layer_sizes": hidden_map[architecture],
            prefix + "alpha": trial.suggest_float(
                "alpha", 1e-6, 1e-1, log=True
            ),
            prefix + "learning_rate_init": trial.suggest_float(
                "learning_rate_init", 1e-4, 1e-2, log=True
            ),
            prefix + "batch_size": trial.suggest_categorical(
                "batch_size", [16, 32, 64]
            ),
        }

    if model_name == "KAN":
        return {
            prefix + "grid_size": trial.suggest_int(
                "grid_size", 3, 7
            ),
            prefix + "spline_order": trial.suggest_int(
                "spline_order", 2, 3
            ),
            prefix + "lr": trial.suggest_float(
                "lr", 3e-4, 5e-3, log=True
            ),
            prefix + "weight_decay": trial.suggest_float(
                "weight_decay", 1e-6, 1e-2, log=True
            ),
        }

    raise KeyError(model_name)

In [ ]:


X_all = df.copy()
y_all = pd.to_numeric(df[TARGET], errors="coerce").astype(float).to_numpy()

def make_outer_splits(X, y, groups_array=None):
    if groups_array is not None:
        n_groups = len(np.unique(groups_array))
        n_splits = min(OUTER_SPLITS, n_groups)
        if n_splits < 3:
            raise ValueError("GroupKFold için en az 3 grup gereklidir.")
        splitter = GroupKFold(n_splits=n_splits)
        return list(splitter.split(X, y, groups_array))

    splitter = RepeatedKFold(
        n_splits=OUTER_SPLITS,
        n_repeats=OUTER_REPEATS,
        random_state=SEED,
    )
    return list(splitter.split(X, y))

OUTER_SPLIT_LIST = make_outer_splits(
    X_all, y_all, groups
)

print(f"Dış fold sayısı: {len(OUTER_SPLIT_LIST)}")
for fold_id, (train_idx, test_idx) in enumerate(
    OUTER_SPLIT_LIST, start=1
):
    if groups is None:
        overlap = "-"
    else:
        overlap = len(
            set(groups[train_idx]) & set(groups[test_idx])
        )
    print(
        f"Fold {fold_id:02d}: train={len(train_idx)}, "
        f"test={len(test_idx)}, grup çakışması={overlap}"
    )

def make_inner_splits(X_train, y_train, groups_train=None):
    if groups_train is not None:
        n_groups = len(np.unique(groups_train))
        n_splits = min(INNER_SPLITS, n_groups)
        if n_splits < 2:
            raise ValueError("Inner GroupKFold için grup sayısı yetersiz.")
        splitter = GroupKFold(n_splits=n_splits)
        return list(
            splitter.split(X_train, y_train, groups_train)
        )

    splitter = KFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=SEED,
    )
    return list(splitter.split(X_train, y_train))

In [ ]:


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def json_safe_params(params):
    safe = {}
    for key, value in params.items():
        if isinstance(value, tuple):
            safe[key] = list(value)
        elif isinstance(value, (np.integer, np.floating)):
            safe[key] = value.item()
        else:
            safe[key] = value
    return safe

def stable_name_seed(text):
    return sum((i + 1) * ord(ch) for i, ch in enumerate(text)) % 10000

def optimize_on_outer_train(
    model_name,
    feature_spec,
    X_train,
    y_train,
    groups_train,
    fold_seed,
    n_trials,
):
    inner_splits = make_inner_splits(
        X_train, y_train, groups_train
    )
    trial_param_cache = {}

    def objective(trial):
        params = sample_hyperparams(trial, model_name)
        trial_param_cache[trial.number] = params
        fold_rmses = []

        for inner_fold, (itr, iva) in enumerate(
            inner_splits, start=1
        ):
            seed = fold_seed + inner_fold
            pipe = make_model_pipeline(
                model_name, feature_spec, seed
            )
            pipe.set_params(**params)

            pipe.fit(
                X_train.iloc[itr],
                y_train[itr],
            )
            pred = pipe.predict(X_train.iloc[iva])
            fold_rmses.append(
                rmse_score(y_train[iva], pred)
            )

            trial.report(
                float(np.mean(fold_rmses)),
                step=inner_fold,
            )
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_rmses))

    sampler = optuna.samplers.TPESampler(
        seed=fold_seed
    )
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=max(2, n_trials // 4),
        n_warmup_steps=1,
    )
    study = optuna.create_study(
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
    )
    study.optimize(
        objective,
        n_trials=n_trials,
        show_progress_bar=False,
        gc_after_trial=True,
    )

    best_params = trial_param_cache[
        study.best_trial.number
    ]
    return best_params, float(study.best_value)

def nested_cv_evaluate(
    model_name,
    feature_set_key,
    X,
    y,
    groups_array=None,
    outer_splits=None,
    n_trials=None,
    stage="final",
):
    if outer_splits is None:
        outer_splits = make_outer_splits(
            X, y, groups_array
        )
    if n_trials is None:
        n_trials = N_TRIALS[model_name]

    feature_spec = FEATURE_SPECS[feature_set_key]
    result_rows = []
    prediction_rows = []

    for fold_id, (train_idx, test_idx) in enumerate(
        outer_splits, start=1
    ):
        fold_seed = (
            SEED
            + 1000 * fold_id
            + stable_name_seed(model_name)
            + stable_name_seed(feature_set_key)
        )
        seed_everything(fold_seed)

        X_train = X.iloc[train_idx].copy()
        X_test = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]

        groups_train = (
            None
            if groups_array is None
            else groups_array[train_idx]
        )

        print(
            f"[{stage}] {feature_set_key} | {model_name} | "
            f"fold {fold_id}/{len(outer_splits)}"
        )
        start = time.time()

        best_params, inner_rmse = optimize_on_outer_train(
            model_name=model_name,
            feature_spec=feature_spec,
            X_train=X_train,
            y_train=y_train,
            groups_train=groups_train,
            fold_seed=fold_seed,
            n_trials=n_trials,
        )

        final_pipe = make_model_pipeline(
            model_name, feature_spec, fold_seed
        )
        final_pipe.set_params(**best_params)
        final_pipe.fit(X_train, y_train)

        pred_train = final_pipe.predict(X_train)
        pred_test = final_pipe.predict(X_test)
        elapsed = time.time() - start

        result_rows.append({
            "stage": stage,
            "feature_set": feature_set_key,
            "model": model_name,
            "outer_fold": fold_id,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "inner_best_rmse": inner_rmse,
            "train_r2": r2_score(y_train, pred_train),
            "test_r2": r2_score(y_test, pred_test),
            "test_rmse": rmse_score(y_test, pred_test),
            "test_mae": mean_absolute_error(y_test, pred_test),
            "elapsed_seconds": elapsed,
            "best_params": json.dumps(
                json_safe_params(best_params),
                ensure_ascii=False,
            ),
        })

        for row_index, true, pred in zip(
            test_idx, y_test, pred_test
        ):
            prediction_rows.append({
                "stage": stage,
                "feature_set": feature_set_key,
                "model": model_name,
                "outer_fold": fold_id,
                "row_index": int(row_index),
                "y_true": float(true),
                "y_pred": float(pred),
                "residual": float(true - pred),
            })

    return (
        pd.DataFrame(result_rows),
        pd.DataFrame(prediction_rows),
    )

def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", text).strip("_")

In [ ]:


RUN_MODE_TAG = "fast" if FAST_MODE else "full"

_config_payload = {
    "code_version": CODE_VERSION,
    "mode": RUN_MODE_TAG,
    "seed": SEED,
    "leakage_policy": LEAKAGE_POLICY,
    "raw_exclusions": RAW_EXCLUSIONS,
    "group_column": GROUP_COLUMN,
    "outer_splits": OUTER_SPLITS,
    "outer_repeats": OUTER_REPEATS,
    "inner_splits": INNER_SPLITS,
    "n_trials": N_TRIALS,
    "ablation_trials": ABLATION_TRIALS,
}
CONFIG_FINGERPRINT = hashlib.md5(
    json.dumps(_config_payload, sort_keys=True).encode("utf-8")
).hexdigest()[:10]
RESULT_TAG = f"{RUN_MODE_TAG}_{CONFIG_FINGERPRINT}"
print(f"[Bilgi] Çalışma yapı etiketi: {RESULT_TAG}")

def checkpoint_paths(stage, feature_set_key, model_name):
    stem = "_".join([
        safe_filename(stage),
        safe_filename(feature_set_key),
        safe_filename(model_name),
        RESULT_TAG,
    ])
    return (
        OUTPUT_DIR / f"{stem}_fold_results.csv",
        OUTPUT_DIR / f"{stem}_predictions.csv",
    )

def run_or_load_nested(
    stage,
    feature_set_key,
    model_name,
    n_trials=None,
):
    result_path, prediction_path = checkpoint_paths(
        stage, feature_set_key, model_name
    )

    if (
        RESUME_FROM_CHECKPOINT
        and result_path.exists()
        and prediction_path.exists()
    ):
        loaded_results = pd.read_csv(result_path)
        loaded_predictions = pd.read_csv(prediction_path)
        expected_folds = len(OUTER_SPLIT_LIST)

        if loaded_results["outer_fold"].nunique() == expected_folds:
            print(
                f"[Checkpoint] Yüklendi: {feature_set_key} | {model_name}"
            )
            return loaded_results, loaded_predictions

        print(
            "[Uyarı] Checkpoint fold sayısı mevcut ayarla uyuşmuyor; "
            "yeniden çalıştırılıyor."
        )

    fold_results, predictions = nested_cv_evaluate(
        model_name=model_name,
        feature_set_key=feature_set_key,
        X=X_all,
        y=y_all,
        groups_array=groups,
        outer_splits=OUTER_SPLIT_LIST,
        n_trials=n_trials or N_TRIALS[model_name],
        stage=stage,
    )

    fold_results.to_csv(result_path, index=False)
    predictions.to_csv(prediction_path, index=False)
    return fold_results, predictions

In [ ]:


ablation_fold_parts = []
ablation_prediction_parts = []

if RUN_ABLATION:
    for feature_set_key in FEATURE_SPECS:
        for model_name in ABLATION_MODELS:
            result_part, prediction_part = run_or_load_nested(
                stage="ablation",
                feature_set_key=feature_set_key,
                model_name=model_name,
                n_trials=ABLATION_TRIALS[model_name],
            )
            ablation_fold_parts.append(result_part)
            ablation_prediction_parts.append(prediction_part)

    ablation_folds = pd.concat(
        ablation_fold_parts, ignore_index=True
    )
    ablation_predictions = pd.concat(
        ablation_prediction_parts, ignore_index=True
    )

    ablation_folds.to_csv(
        OUTPUT_DIR / f"ablation_all_fold_results_{RESULT_TAG}.csv",
        index=False,
    )
    ablation_predictions.to_csv(
        OUTPUT_DIR / f"ablation_all_predictions_{RESULT_TAG}.csv",
        index=False,
    )
else:
    ablation_folds = pd.DataFrame()
    ablation_predictions = pd.DataFrame()
    print("[Bilgi] RUN_ABLATION=False; ablation çalıştırılmadı.")

In [ ]:
import os

directory_path = "/content/drive/MyDrive/barley_project"

if os.path.exists(directory_path):
    print(f"Files in {directory_path}:")
    for filename in os.listdir(directory_path):
        print(filename)
else:
    print(f"Directory not found: {directory_path}")

In [ ]:


def summarize_fold_results(fold_df):
    if fold_df.empty:
        return pd.DataFrame()

    summary = (
        fold_df
        .groupby(["feature_set", "model"], as_index=False)
        .agg(
            R2_mean=("test_r2", "mean"),
            R2_sd=("test_r2", "std"),
            RMSE_mean=("test_rmse", "mean"),
            RMSE_sd=("test_rmse", "std"),
            MAE_mean=("test_mae", "mean"),
            MAE_sd=("test_mae", "std"),
            Train_R2_mean=("train_r2", "mean"),
            Fold_count=("outer_fold", "nunique"),
            Time_seconds=("elapsed_seconds", "sum"),
        )
    )
    summary["Overfit_gap"] = (
        summary["Train_R2_mean"] - summary["R2_mean"]
    )
    return summary.sort_values(
        ["feature_set", "R2_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)

ablation_summary = summarize_fold_results(ablation_folds)
display(ablation_summary)

if not ablation_summary.empty:
    ablation_summary.to_csv(
        OUTPUT_DIR / f"ablation_summary_{RESULT_TAG}.csv",
        index=False,
    )

ADJACENT_ABLATIONS = [
    ("A0_REF9_SAFE", "A1_ALL14_SAFE"),
    ("A1_ALL14_SAFE", "A2_SAFE_ENGINEERED"),
    ("A2_SAFE_ENGINEERED", "A3_SAFE_ENGINEERED_PCAKMEANS"),
]

ablation_test_rows = []

if not ablation_folds.empty:
    for model_name in ABLATION_MODELS:
        model_df = ablation_folds[
            ablation_folds["model"] == model_name
        ]

        for left_set, right_set in ADJACENT_ABLATIONS:
            left = (
                model_df[model_df["feature_set"] == left_set]
                .set_index("outer_fold")["test_r2"]
            )
            right = (
                model_df[model_df["feature_set"] == right_set]
                .set_index("outer_fold")["test_r2"]
            )
            paired = pd.concat(
                [left.rename("left"), right.rename("right")],
                axis=1,
            ).dropna()

            differences = paired["right"] - paired["left"]
            if len(paired) < 3 or np.allclose(differences, 0):
                statistic, p_value = 0.0, 1.0
            else:
                statistic, p_value = wilcoxon(
                    paired["right"],
                    paired["left"],
                    alternative="two-sided",
                    zero_method="wilcox",
                )

            ablation_test_rows.append({
                "model": model_name,
                "left_feature_set": left_set,
                "right_feature_set": right_set,
                "mean_R2_change_right_minus_left": differences.mean(),
                "median_R2_change": differences.median(),
                "wilcoxon_statistic": statistic,
                "p_raw": p_value,
                "n_paired_folds": len(paired),
            })

ablation_tests = pd.DataFrame(ablation_test_rows)

if not ablation_tests.empty:
    _, p_holm, _, _ = multipletests(
        ablation_tests["p_raw"],
        alpha=0.05,
        method="holm",
    )
    ablation_tests["p_holm"] = p_holm
    ablation_tests["significant_holm_0_05"] = (
        ablation_tests["p_holm"] < 0.05
    )
    display(ablation_tests)
    ablation_tests.to_csv(
        OUTPUT_DIR / f"ablation_wilcoxon_holm_{RESULT_TAG}.csv",
        index=False,
    )

In [ ]:


if not ablation_summary.empty:
    feature_order = list(FEATURE_SPECS.keys())
    fig, ax = plt.subplots(figsize=(11, 6))

    for model_name in ABLATION_MODELS:
        part = (
            ablation_summary[
                ablation_summary["model"] == model_name
            ]
            .set_index("feature_set")
            .reindex(feature_order)
        )
        ax.errorbar(
            feature_order,
            part["R2_mean"],
            yerr=part["R2_sd"],
            marker="o",
            capsize=4,
            label=model_name,
        )

    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("Dış fold test R² (ortalama ± SD)")
    ax.set_xlabel("Kontrollü özellik aşaması")
    ax.set_title("Kontrollü Ablation: Her Aşamada Tek Bileşen Eklenmesi")
    ax.tick_params(axis="x", rotation=20)
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"ablation_R2_{RESULT_TAG}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:


final_fold_parts = []
final_prediction_parts = []

print(f"[Bilgi] Nihai özellik seti: {FINAL_FEATURE_SET_KEY}")
print(f"[Bilgi] Model sayısı: {len(CORE_MODELS)}")

if RUN_FINAL_MODELS:
    for model_name in CORE_MODELS:
        result_part, prediction_part = run_or_load_nested(
            stage="final",
            feature_set_key=FINAL_FEATURE_SET_KEY,
            model_name=model_name,
            n_trials=N_TRIALS[model_name],
        )
        final_fold_parts.append(result_part)
        final_prediction_parts.append(prediction_part)

    final_folds = pd.concat(
        final_fold_parts, ignore_index=True
    )
    final_predictions = pd.concat(
        final_prediction_parts, ignore_index=True
    )

    final_folds.to_csv(
        OUTPUT_DIR / f"final_all_fold_results_{RESULT_TAG}.csv",
        index=False,
    )
    final_predictions.to_csv(
        OUTPUT_DIR / f"final_all_predictions_{RESULT_TAG}.csv",
        index=False,
    )
else:
    final_folds = pd.DataFrame()
    final_predictions = pd.DataFrame()
    print("[Bilgi] RUN_FINAL_MODELS=False; nihai modeller çalıştırılmadı.")

In [ ]:


def bootstrap_mean_ci(values, n_boot=5000, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    boot_means = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_means[i] = sample.mean()

    return np.percentile(boot_means, [2.5, 97.5])

final_summary_rows = []

if not final_folds.empty:
    for model_name, part in final_folds.groupby("model"):
        r2_ci = bootstrap_mean_ci(part["test_r2"])
        rmse_ci = bootstrap_mean_ci(part["test_rmse"])

        final_summary_rows.append({
            "Model": model_name,
            "R2_mean": part["test_r2"].mean(),
            "R2_sd": part["test_r2"].std(ddof=1),
            "R2_CI_low": r2_ci[0],
            "R2_CI_high": r2_ci[1],
            "RMSE_mean": part["test_rmse"].mean(),
            "RMSE_sd": part["test_rmse"].std(ddof=1),
            "RMSE_CI_low": rmse_ci[0],
            "RMSE_CI_high": rmse_ci[1],
            "MAE_mean": part["test_mae"].mean(),
            "MAE_sd": part["test_mae"].std(ddof=1),
            "Train_R2_mean": part["train_r2"].mean(),
            "Overfit_gap": (
                part["train_r2"].mean()
                - part["test_r2"].mean()
            ),
            "Fold_count": part["outer_fold"].nunique(),
            "Total_time_seconds": part["elapsed_seconds"].sum(),
        })

final_summary = (
    pd.DataFrame(final_summary_rows)
    .sort_values("R2_mean", ascending=False)
    .reset_index(drop=True)
)

display(final_summary)

if not final_summary.empty:
    final_summary.to_csv(
        OUTPUT_DIR / f"final_model_summary_{RESULT_TAG}.csv",
        index=False,
    )

In [ ]:


def rank_biserial_from_differences(differences):
    differences = np.asarray(differences, dtype=float)
    differences = differences[~np.isclose(differences, 0)]
    if len(differences) == 0:
        return 0.0

    ranks = rankdata(np.abs(differences))
    positive = ranks[differences > 0].sum()
    negative = ranks[differences < 0].sum()
    return float((positive - negative) / (positive + negative))

friedman_result = pd.DataFrame()
average_ranks = pd.DataFrame()
posthoc_results = pd.DataFrame()

if not final_folds.empty:
    score_matrix = final_folds.pivot(
        index="outer_fold",
        columns="model",
        values="test_r2",
    ).dropna(axis=0, how="any")

    if score_matrix.shape[0] < 3:
        raise ValueError("Friedman testi için yeterli eşleştirilmiş fold yok.")

    statistic, p_value = friedmanchisquare(
        *[score_matrix[col].to_numpy() for col in score_matrix.columns]
    )
    friedman_result = pd.DataFrame([{
        "metric": "test_r2",
        "n_blocks": score_matrix.shape[0],
        "n_models": score_matrix.shape[1],
        "friedman_chi_square": statistic,
        "p_value": p_value,
        "significant_0_05": p_value < 0.05,
    }])
    display(friedman_result)

    fold_ranks = score_matrix.rank(
        axis=1,
        ascending=False,
        method="average",
    )
    average_ranks = (
        fold_ranks.mean(axis=0)
        .sort_values()
        .rename("average_rank")
        .reset_index()
        .rename(columns={"model": "Model"})
    )
    display(average_ranks)

    pair_rows = []
    for model_a, model_b in combinations(score_matrix.columns, 2):
        x = score_matrix[model_a].to_numpy()
        y = score_matrix[model_b].to_numpy()
        differences = x - y

        if np.allclose(differences, 0):
            stat, raw_p = 0.0, 1.0
        else:
            stat, raw_p = wilcoxon(
                x,
                y,
                alternative="two-sided",
                zero_method="wilcox",
            )

        pair_rows.append({
            "Model_A": model_a,
            "Model_B": model_b,
            "Mean_R2_A_minus_B": float(np.mean(differences)),
            "Median_R2_A_minus_B": float(np.median(differences)),
            "Rank_biserial_A_vs_B": rank_biserial_from_differences(
                differences
            ),
            "Wilcoxon_statistic": stat,
            "p_raw": raw_p,
            "n_paired_folds": len(differences),
        })

    posthoc_results = pd.DataFrame(pair_rows)
    _, p_holm, _, _ = multipletests(
        posthoc_results["p_raw"],
        alpha=0.05,
        method="holm",
    )
    posthoc_results["p_holm"] = p_holm
    posthoc_results["significant_holm_0_05"] = (
        posthoc_results["p_holm"] < 0.05
    )
    posthoc_results = posthoc_results.sort_values(
        ["p_holm", "p_raw"]
    ).reset_index(drop=True)
    display(posthoc_results)

    friedman_result.to_csv(
        OUTPUT_DIR / f"friedman_test_{RESULT_TAG}.csv",
        index=False,
    )
    average_ranks.to_csv(
        OUTPUT_DIR / f"average_model_ranks_{RESULT_TAG}.csv",
        index=False,
    )
    posthoc_results.to_csv(
        OUTPUT_DIR / f"wilcoxon_posthoc_holm_{RESULT_TAG}.csv",
        index=False,
    )

In [ ]:


if not final_folds.empty:
    model_order = (
        final_summary.sort_values("R2_mean", ascending=False)["Model"]
        .tolist()
    )

    box_data = [
        final_folds.loc[
            final_folds["model"] == model, "test_r2"
        ].to_numpy()
        for model in model_order
    ]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.boxplot(
        box_data,
        labels=model_order,
        showmeans=True,
    )
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.set_ylabel("Dış fold test R²")
    ax.set_xlabel("Model")
    ax.set_title(
        f"Nested CV Model Karşılaştırması — {FINAL_FEATURE_SET_KEY}"
    )
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"final_model_R2_boxplot_{RESULT_TAG}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

if not final_predictions.empty and not final_summary.empty:
    best_model = final_summary.iloc[0]["Model"]
    best_pred = final_predictions[
        final_predictions["model"] == best_model
    ].copy()

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ax.scatter(
        best_pred["y_true"],
        best_pred["y_pred"],
        alpha=0.65,
    )
    low = min(
        best_pred["y_true"].min(),
        best_pred["y_pred"].min(),
    )
    high = max(
        best_pred["y_true"].max(),
        best_pred["y_pred"].max(),
    )
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("Gerçek HI")
    ax.set_ylabel("Dış fold tahmini HI")
    ax.set_title(f"Gerçek–Tahmin: {best_model}")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(
        OUTPUT_DIR / f"best_model_actual_vs_predicted_{RESULT_TAG}.png",
        dpi=200,
        bbox_inches="tight",
    )
    plt.show()

In [ ]:


RUN_SYNTHETIC_APPENDIX = False

if RUN_SYNTHETIC_APPENDIX:
    raise NotImplementedError(
        "Sentetik veri deneyi ana analizden bilinçli olarak ayrılmıştır. "
        "Ayrı bir notebook ve önceden tanımlanmış bağımsız protokol kullanın."
    )
else:
    print(
        "[Bilgi] Sentetik veri üretimi ana analizde çalıştırılmadı. "
        "Bu beklenen ve yöntemsel olarak tercih edilen davranıştır."
    )

In [ ]:


version_packages = [
    "numpy", "pandas", "scikit-learn", "torch",
    "xgboost", "catboost", "optuna", "statsmodels",
]
version_rows = []
for package_name in version_packages:
    try:
        version = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        version = "not-installed"
    version_rows.append({
        "package": package_name,
        "version": version,
    })

environment_df = pd.DataFrame(version_rows)
environment_df.to_csv(
    OUTPUT_DIR / "environment_versions.csv",
    index=False,
)

run_manifest = {
    "CODE_VERSION": CODE_VERSION,
    "FAST_MODE": FAST_MODE,
    "RUN_MODE_TAG": RUN_MODE_TAG,
    "RESULT_TAG": RESULT_TAG,
    "SEED": SEED,
    "TARGET": TARGET,
    "DATA_PATH": str(DATA_PATH),
    "LEAKAGE_POLICY": LEAKAGE_POLICY,
    "RAW_EXCLUSIONS": RAW_EXCLUSIONS,
    "GROUP_COLUMN": GROUP_COLUMN,
    "FINAL_FEATURE_SET_KEY": FINAL_FEATURE_SET_KEY,
    "OUTER_SPLITS": OUTER_SPLITS,
    "OUTER_REPEATS": OUTER_REPEATS,
    "INNER_SPLITS": INNER_SPLITS,
    "CORE_MODELS": CORE_MODELS,
    "ABLATION_MODELS": ABLATION_MODELS,
    "N_TRIALS": N_TRIALS,
    "ABLATION_TRIALS": ABLATION_TRIALS,
}

with open(
    OUTPUT_DIR / f"run_manifest_{RESULT_TAG}.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        run_manifest,
        f,
        ensure_ascii=False,
        indent=2,
    )

archive_base = str(
    OUTPUT_DIR.parent / f"{OUTPUT_DIR_NAME}_{RESULT_TAG}"
)
zip_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=OUTPUT_DIR,
)

print(f"[Tamamlandı] Çıktı klasörü: {OUTPUT_DIR}")
print(f"[Tamamlandı] ZIP arşivi   : {zip_path}")

if IN_COLAB:
    from google.colab import files
    print(
        "ZIP Drive'a kaydedildi. Tarayıcıya indirmek için "
        "aşağıdaki satırın yorumunu kaldırabilirsiniz:"
    )
    print(f"# files.download(r'{zip_path}')")